In [2]:
import pandas as pd

df = pd.read_json(r"C:\Users\jorge leandro\OneDrive\Desktop\proyecto_final_md\data\raw\streaming_users_dirty.json")
df_procesado = df.copy()

Tratamiento de valores faltantes

In [3]:
#Variable numérica monthly_watch_time_mins 
print("Media:", df_procesado["monthly_watch_time_mins"].mean())
print("Mediana:", df_procesado["monthly_watch_time_mins"].median())

Media: 1107.3461528806326
Mediana: 757.4


La media es mucho mayor que la mediana (hay una diferencia de casi 350 minutos). Esto indica que la variable no tiene una distribución aproximadamente normal y probablemente esté sesgada hacia la derecha (hay algunos usuarios con tiempos de visualización muy altos que elevan la media).

¿Qué conviene hacer?

En este caso, lo recomendable es imputar los valores faltantes con la mediana, porque:

La mediana no se ve afectada por valores extremos.
Representa mejor el valor central cuando la distribución está sesgada.

In [4]:
df_procesado["monthly_watch_time_mins"] = df_procesado["monthly_watch_time_mins"].fillna(
    df_procesado["monthly_watch_time_mins"].median())

In [5]:
#Variable categórica favorite_genre
df_procesado["favorite_genre"] = df_procesado["favorite_genre"].fillna("Desconocido")

Al tratarse de una variable categórica, asignar una categoría explícita evita la pérdida de registros y permite identificar posteriormente aquellos usuarios cuyo género favorito no fue informado

In [6]:
#Variable temporal last_login_date
#Conversión de object a datetime
df_procesado["last_login_date"] = pd.to_datetime(
    df_procesado["last_login_date"],
    errors="coerce")

In [7]:
df_procesado["last_login_date"].isnull().sum()

np.int64(769)

In [10]:
df = pd.read_json(r"C:\Users\jorge leandro\OneDrive\Desktop\proyecto_final_md\data\raw\streaming_users_dirty.json")
fechas = pd.to_datetime(
    df["last_login_date"],
    errors="coerce"
)

df.loc[fechas.isna(), "last_login_date"].value_counts(dropna=False)

last_login_date
None             320
2026-15-03        20
0000-00-00        17
not_available     14
31-02-2022        13
                ... 
03-16-2022         1
08-15-2022         1
01-11-2024         1
02-08-2018         1
2019/10/09         1
Name: count, Length: 380, dtype: int64

La columna last_login_date presentaba 320 valores nulos originales. Además, al convertirla al tipo datetime, se detectaron 409 valores adicionales con fechas inválidas o formatos no reconocidos (0000-00-00, 31-02-2022, 2026-15-03, not_available, entre otros), alcanzando un total de 729 valores faltantes (NaT).

Se convirtió la columna al tipo datetime utilizando pd.to_datetime(errors="coerce"). Los valores inválidos se representaron como NaT.

No se imputaron fechas debido a que la fecha de último inicio de sesión representa un dato histórico específico. Asignar una fecha artificial (por ejemplo, la media o la mediana) introduciría información que no existe y podría afectar los análisis temporales.

La columna quedó correctamente tipada como datetime64[ns], identificando de forma explícita todas las fechas faltantes o inválidas y preservando la integridad del dataset.

Tratamiento de valores duplicados

In [11]:
#Valores duplicados
df_procesado.duplicated().sum()

np.int64(126)

In [20]:
df_procesado[df_procesado.duplicated(keep=False)].sort_values("user_id")

,user_id,age,subscription_plan,monthly_watch_time_mins,country,favorite_genre,last_login_date,customer_support_tickets
37,10037,33,Básico,611.0,Colombia,Documental,2019-09-29,2
8133,10037,33,Básico,611.0,Colombia,Documental,2019-09-29,2
8089,10052,25,Básico,465.7,Colombia,Acción,2022-03-31,1
52,10052,25,Básico,465.7,Colombia,Acción,2022-03-31,1
117,10117,29,Estándar,713.9,Brasil,Documental,2020-12-20,1
...,...,...,...,...,...,...,...,...
8115,17684,38,Premium,1283.2,México,Comedia,2024-09-13,0
8125,17784,38,Estándar,492.2,México,Documental,2021-10-04,1
7784,17784,38,Estándar,492.2,México,Documental,2021-10-04,1
7994,17994,24,Básico,437.0,México,Romance,2020-08-12,0


In [22]:
df_procesado = df_procesado.drop_duplicates()
df_procesado.duplicated().sum()

np.int64(0)

In [23]:
#Verificación del estado del dataset
df_procesado.info()
df_procesado.isnull().sum()
df_procesado.duplicated().sum()

<class 'pandas.core.frame.DataFrame'>
Index: 8034 entries, 0 to 8158
Data columns (total 8 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   user_id                   8034 non-null   int64         
 1   age                       8034 non-null   int64         
 2   subscription_plan         8034 non-null   object        
 3   monthly_watch_time_mins   8034 non-null   float64       
 4   country                   8034 non-null   object        
 5   favorite_genre            8034 non-null   object        
 6   last_login_date           7265 non-null   datetime64[ns]
 7   customer_support_tickets  8034 non-null   int64         
dtypes: datetime64[ns](1), float64(1), int64(3), object(3)
memory usage: 564.9+ KB


np.int64(0)

In [24]:
df_procesado.describe()

,user_id,age,monthly_watch_time_mins,last_login_date,customer_support_tickets
count,8034.000000,8034.000000,8034.000000,7265,8034.000000
mean,13999.920961,34.102315,1103.283321,2022-02-04 06:52:40.467997184,1.817028
min,10000.000000,-5.000000,-120.000000,2018-01-01 00:00:00,-1.000000
25%,11999.250000,25.000000,493.725000,2020-01-24 00:00:00,0.000000
50%,14001.500000,33.000000,757.400000,2022-02-15 00:00:00,1.000000
75%,15997.750000,42.000000,1035.600000,2024-02-17 00:00:00,1.000000
max,17999.000000,150.000000,99999.000000,2029-01-01 00:00:00,150.000000
std,2308.946138,14.563588,5288.188527,NaN,11.422279


In [25]:
df_procesado[df_procesado["age"] < 0]
df_procesado[df_procesado["age"] > 100]

,user_id,age,subscription_plan,monthly_watch_time_mins,country,favorite_genre,last_login_date,customer_support_tickets
194,10194,130,Básico,820.7,Uruguay,Drama,2018-10-11,1
324,10324,130,Básico,420.8,Brasil,Thriller,2023-05-07,1
426,10426,150,Premium,1315.6,colombia,ACCIÓN,2022-12-15,1
529,10529,130,Básico,590.1,Uruguay,Thriller,2018-10-14,0
640,10640,150,Premium,1374.6,Argentina,Drama,2025-02-24,2
655,10655,130,Premium,1097.5,Chile,Documental,2018-11-27,0
751,10751,150,Básico,167.4,Chile,Acción,2019-04-14,0
1028,11028,150,Básico,302.7,Argentina,Acción,2018-06-13,1
1099,11099,150,Básico,771.3,Perú,Romance,2021-03-25,-1
1374,11374,130,Básico,757.4,Uruguay,Acción,2024-04-18,0


In [26]:
df_procesado[df_procesado["monthly_watch_time_mins"] < 0]

,user_id,age,subscription_plan,monthly_watch_time_mins,country,favorite_genre,last_login_date,customer_support_tickets
443,10443,31,Estándar,-120.0,Chile,Thriller,2019-06-07,1
797,10797,31,Básico,-1.0,México,Comedia,2023-07-01,1
1126,11126,0,Premium,-1.0,Colombia,Documental,2021-01-24,1
1186,11186,35,Premium,-120.0,Brasil,Drama,2022-12-09,0
1340,11340,40,Estándar,-120.0,Perú,Documental,2023-05-28,2
1466,11466,13,Estándar,-120.0,Perú,Acción,2018-03-02,1
1725,11725,28,Estándar,-120.0,Perú,Comedia,2022-06-29,0
1843,11843,38,Básico,-120.0,Argentina,Action,2025-08-10,1
2318,12318,29,Estándar,-1.0,CHL,Documental,2018-05-05,1
2384,12384,33,Estándar,-120.0,Uruguay,Desconocido,2020-09-13,1


In [27]:
df_procesado["country"].value_counts()
df_procesado["subscription_plan"].value_counts()
df_procesado["favorite_genre"].value_counts()

favorite_genre
Comedia        1095
Drama          1088
Thriller       1077
Documental     1070
Romance        1070
Acción         1066
Crime          1049
Desconocido     240
Action           22
COMEDIA          19
Crimen           18
Romance          17
CRIME            17
Documentary      16
DRAMA            16
Comedia          15
DOC              15
ROMANCE          14
ACCIÓN           14
THRILLER         14
comedy           12
Thriller         12
romance          12
documental       10
drama            10
accion            8
Drama             7
thriler           6
crime             5
Name: count, dtype: int64